# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import openpyxl
from itertools import islice
from pathlib import Path
import importlib
import utils  
importlib.reload(utils)
from utils import * 

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

### Inputs and Paths

In [2]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/NCBI_Virus_Andersen_GISAID/" 

# Collect user input

# locations = input("Locations (separate with commas and no spaces): ")
# start_date = input("Start date (format: YYYY-MM-DD): ")
# end_date = input("End date (format: YYYY-MM-DD): ")

locations = "Antarctica,North America,South America"
genotypes = ["B3.13"] 
start_date = "2021-11-01"
end_date = "2026-07-17"
date_range = dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y")

os.chdir(downloads)

# Create directories if needed
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen_ncbi_virus = home + "NCBI_Virus/complete/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
# andersen_ncbi_virus = home + "Combinations/NCBI_Virus_Andersen/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
complete_files = andersen_ncbi_virus_gisaid + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# Get list of genotypes and states

os.chdir(references)

states = pd.read_csv("states_ref.csv")



## Download all files, convert fasta files to dataframes

In [3]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
            try:
                shutil.move(file_name, destination_path)
            except:
                print("Error moving file", file_name)
                continue 
    # elif len(files) == 0 and len(os.listdir(downloads_saved)) == 0: # If we don't have any downloaded files
    #     # Have user type in username and password
    #     username = input("Username: ")
    #     password = input("Password: ")
    #     browser = input("Browser: ")
    #     sleep_time = input("Seconds to sleep in between clicks (recommended 5): ")

    #     open_gisaid(username, password, browser, sleep_time, locations, start_date, end_date) # Download files -- NOT WORKING RIGHT NOW
    #     # First batch: 2021-11-01 -- 2024-12-31
    #     # Second batch: 2025-01-01 -- present

    #     # Re-try 
    #     for dirpath, dirs, files in os.walk(downloads):
    #         if len(files) > 0: # If we have any files that need to be moved
    #             for file in files:
    #                 file_name = os.path.join(dirpath, file)
    #                 destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
    #                 try:
    #                     shutil.move(file_name, destination_path)
    #                 except:
    #                     print("Error moving file", file_name)
    #                     continue 
    #         break 
    else: # If we have downloaded files saved already
        continue
    break 

for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # file_name = "_".join(file_name.split(" "))
        
        
        # os.rename(file_name, "_".join(file_name.split(" ")).replace("(", "").replace(")", ""))
        # file_name = "_".join(file_name.split(" ")).replace("(", "").replace(")", "")

        print(file_name)

        # Now go through files and get contents
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)
    break 

C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-07-17_Antarctica_North_America_South_America/gisaid_epiflu_isolates (1).xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-07-17_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-07-17_Antarctica_North_America_South_America/gisaid_epiflu_sequence (1).fasta
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-07-17_Antarctica_North_America_South_America/gisaid_epiflu_sequence.fasta


In [4]:
# Concatenate metadata

metadata_concat = pd.DataFrame()
for metadata_file in all_metadata_files:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

In [5]:
# Function to get metadata
def separate_fasta_by_segments(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first

    unique_segments = list(set(fasta["Segment"])) # Get list of segments

    # “>EPI_ID|Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for genotype in genotypes: # .keys(): # For each genotype
        for seg in unique_segments: # For each segment
            print(list(set(metadata["Genotype"])))
            xls = metadata[(metadata["Genotype"].str.contains(genotype)) & metadata["Publishing_Embargo_Until"].isna()] # [metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == genotype] # Get only the metadata corresponding to that genotype
            xls = xls.rename(columns={"Isolate_Id":"Identifier"})
            # print(xls)
            # print("XLS: ", metadata["Genotype"])
            # print(xls["Clade"])

            # fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
            fasta_seg_pre = fasta.merge(xls, how="right", on="Identifier")
            # print(fasta_seg_pre.columns)
            # break 

            # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = genotype

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name_x"] + "|" + fasta_seg["Subtype_x"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] #.apply(lambda x: "" if x != "human" else "|human")
            fasta_seg["full_header"] = new_name
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            print(fasta_seg[["full_header", "Header", "Isolate_Id", "Isolate_Name_x", "Subtype_x", "Segment", "Geo_Location", "Date Collected", "Identifier", "Host_Type", "Isolate_Name_y", "Subtype_y", "Genotype", "Location", "Collection_Date"]])

            # segment_fastas.append(fasta_seg)

    return segment_fastas, unique_segments

In [20]:
# Separate fastas by segment -- results in number of downloaded fastas * number of genotypes * 8 segments
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):

    metadata = metadata_concat

    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segments(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment

    for fasta in fastas:
        segment_fastas.append(fasta)

# print(segment_fastas) # [0][0][segment_fastas[0][0]["Genotype"] == "D1.1"])

["B1.2 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>)", "Minor13 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / E.5 (<i style='font-size:11px'>ggFLU</i>)", "C2.1 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>)", "Notassigned (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / L.1 (<i style='font-size:11px'>ggFLU</i>)", "Minor105 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / P.5 (<i style='font-size:11px'>ggFLU</i>)", "Minor97 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>)", "B3.10 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / K.1 (<i style='font-size:11px'>ggFLU</i>)", "B3.5 (<i style='font-size:11px'>GenoFLU</i>) / unassigned (<i style='font-size:11px'>Genin</i>) / E.2 (<i style

In [21]:
print(len(segment_fastas))

16


## De-Duplication

In [22]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {} # Results in number of genotypes * 8 segments
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
        # print(file_name)
            segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
            fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
            fasta_file["full_header"] = fasta_file["Header"].apply(lambda x: ">" + x)
            andersen_ncbi[segment_genotype] = fasta_file
    break 

In [23]:
print(andersen_ncbi)

{'B3.13_HA':                                                  Header  \
0     SRR30811263|A/Turkey/CA/24-027086-001-original...   
1     SRR30811262|A/Turkey/CA/24-027086-002-original...   
2     SRR30811260|A/Turkey/CA/24-027086-003-original...   
3     SRR30811259|A/Turkey/CA/24-027086-004-original...   
4     SRR30811249|A/cattle/CA/24-027102-001-original...   
...                                                 ...   
4647  GCA_039158345.1|A/Bovine/texas/24-029328-01/20...   
4648  GCA_039158355.1|A/bovine/texas/24-029328-02/20...   
4649  GCA_039158365.1|A/feline/Texas/24-029329-01/20...   
4650  GCA_039158375.1|A/feline/Texas/24-029329-02/20...   
4651  GCA_039465435.1|A/Texas/37/2024|H5N1|USA-TX|20...   

                  Isolate_Id                             Isolate_Name Subtype  \
0     24-027086-001-original  A/Turkey/CA/24-027086-001-original/2024    H5N1   
1     24-027086-002-original  A/Turkey/CA/24-027086-002-original/2024    H5N1   
2     24-027086-003-original  A/Tur

### Collect partial isolates from all parties

In [24]:
# Do all segments, not just HA 

# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

# GISAID partial isolates
gisaid_list = {}
for gisaid_fasta in segment_fastas:
    if len(gisaid_fasta) > 0:
        # for genotype_gisaid_fasta in gisaid_fasta:
        # print(genotype_gisaid_fasta)
        gisaid_fasta["Partials"] = gisaid_fasta["Isolate_Id"].apply(partial_isolate)
        gisaid_fasta["Year"] = gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
        genotype_gisaid_fasta = gisaid_fasta.drop_duplicates(subset=["Partials", "Year", "Segment"], keep="first")
        # print(genotype_gisaid_fasta["Genotype"])
        if genotype_gisaid_fasta["Genotype"].values[0].split(" ")[0] not in gisaid_list.keys():
            gisaid_list[genotype_gisaid_fasta["Genotype"].values[0].split(" ")[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]] = genotype_gisaid_fasta
        else:
            gisaid_list[genotype_gisaid_fasta["Genotype"].values[0].split(" ")[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]] = pd.concat([gisaid_list[genotype_gisaid_fasta["Genotype"].values[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]], genotype_gisaid_fasta]).drop_duplicates(subset=["Partials", "Year", "Segment"], keep="last")
        
        # print(genotype_gisaid_fasta)
        
# NCBI_Virus/Andersen partial isolates    
andersen_ncbi_genotypes = {} # Results in # of genotypes
for key in andersen_ncbi:
    print(key)
    andersen_ncbi_fasta = andersen_ncbi[key] 

    # Find partial Isolate IDs -- humans and non-humans have different locations for isolates
    andersen_ncbi_fasta_nonhuman = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] != "human"]
    andersen_ncbi_fasta_nonhuman["Partials"] = andersen_ncbi_fasta_nonhuman["Isolate_Id"].apply(partial_isolate) # For non-humans, apply partial function to isolate ids

    andersen_ncbi_fasta_human = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] == "human"]
    # andersen_ncbi_fasta_human["Partials_Prep"] = andersen_ncbi_fasta_human["Isolate_Name"].apply(lambda x: x.split("/")[-2]) # For humans, apply partial function to location, because isolate comes earlier
    andersen_ncbi_fasta_human["Partials"] = andersen_ncbi_fasta_human["Isolate_Id"].apply(partial_isolate)
    # andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
    # print("Andersen:", andersen_ncbi_fasta["Partials"])

    # Concatenate humans and non-humans
    andersen_ncbi_fasta = pd.concat([andersen_ncbi_fasta_human, andersen_ncbi_fasta_nonhuman])
    # print(andersen_ncbi_fasta)

    # Get year and segment
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if len(x) > 0 else x)
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
    # andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    # print(andersen_ncbi_fasta)

    if andersen_ncbi_fasta["Genotype"].values[0].split(" ")[0] not in andersen_ncbi_genotypes.keys(): # If we haven't already seen this genotype
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0].split(" ")[0].split("_")[0] + "_" + andersen_ncbi_fasta["Segment"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype (To include "Not" assigned) dictionary
        # print(andersen_ncbi_fasta)
    # break 


B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2


In [25]:
# print(andersen_ncbi_genotypes["Not_PB2"]) #["Segment"].values)

In [26]:
for gisaid_df in gisaid_list:
    print(gisaid_df)
    print(gisaid_list[gisaid_df])
    print(len(gisaid_df))

print(len(gisaid_list))

B3.13_NP
                                                  Header         Isolate_Id  \
69     EPI_ISL_19497981|A/California/152/2024|A_/_H5N...                152   
74     EPI_ISL_19497980|A/California/153/2024|A_/_H5N...                153   
85     EPI_ISL_19532150|A/chicken/USA/24-031862-002/2...      24-031862-002   
96     EPI_ISL_19532151|A/dairy_cow/California/031313...         031313-004   
101    EPI_ISL_19532152|A/dairy_cow/California/031313...         031313-003   
...                                                  ...                ...   
38339  EPI_ISL_19332142|A/american_robin/Iowa/24-0188...      24-018880-001   
38354  EPI_ISL_19628008|A/California/213/2024|A_/_H5N...                213   
38365  EPI_ISL_19332143|A/american_robin/Iowa/24-0188...      24-018879-003   
38373  EPI_ISL_19167901|A/chicken/Michigan/24-010308-...  24-010308-006-300   
38381  EPI_ISL_19167900|A/chicken/Michigan/24-010308-...  24-010308-001-300   

                                  Isolate_

### Throw away duplicate isolates from GISAID

In [27]:
gisaid_dict = {}

for gisaid_genotype in gisaid_list: 
    gisaid_genotype_df = gisaid_list[gisaid_genotype]
    print("original:", len(gisaid_genotype_df))
    print(gisaid_genotype)
    if gisaid_genotype in andersen_ncbi_genotypes.keys(): # If they share the genotype
        # Left inner on gisaid, so we can get all duplicates and ignore unique Andersen entries
        gisaid_duplicates = pd.merge(gisaid_genotype_df, andersen_ncbi_genotypes[gisaid_genotype], how="inner") #, indicator=True) #, join="left") #, on=shared_columns)
        # print(gisaid_duplicates)
        gisaid_deduplicated = pd.concat([gisaid_genotype_df, gisaid_duplicates]).drop_duplicates(subset=["Partials"], keep="first") # Use Andersen/NCBI_Virus duplicates instead of GISAID
        gisaid_dict[gisaid_genotype] = gisaid_deduplicated
        print("new:", len(gisaid_deduplicated))
    else: # If this is a GISAID-only genotype
        gisaid_deduplicated = gisaid_genotype_df.drop_duplicates(subset=["Partials"], keep="last")
        gisaid_dict[gisaid_genotype] = gisaid_deduplicated
        print("new:", len(gisaid_deduplicated))
            

# throw_away = {} # Dictionary of isolates to throw

# counter = 0
# for gisaid_genotype in gisaid_list: # Each dataframe is unique in genotype
#     # print(gisaid_genotype)
#     if len(gisaid_genotype["Genotype"]) > 0:
#         genotype = gisaid_genotype["Genotype"].values[0]
#         andersen_ncbi_fasta = pd.DataFrame()
#         if genotype in andersen_ncbi_genotypes.keys(): # If it's in Andersen/NCBI_Virus and we haven't seen it before here
#             andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
#             gisaid_duplicates = gisaid_genotype[gisaid_genotype.duplicated(["Partials", "Year"], keep=False)]
#             # print(gisaid_duplicates)
#             andersen_ncbi_fasta_duplicates = andersen_ncbi_fasta[andersen_ncbi_fasta.duplicated(["Partials", "Year"], keep=False)]
#             to_throw = pd.concat([gisaid_genotype, andersen_ncbi_fasta])[pd.concat([gisaid_genotype, andersen_ncbi_fasta]).duplicated(["Partials", "Year"], keep=False)]
            
#             between_duplicates = []
#             for t in to_throw["Identifier"].values:
#                 if t not in gisaid_duplicates and t not in andersen_ncbi_fasta_duplicates:
#                 # if t in gisaid_duplicates or t in andersen_ncbi_fasta_duplicates:
#                     # print(t)
#                     between_duplicates.append(t)
#             if genotype in throw_away.keys(): # If there's already a genotype
#                 throw_away[genotype] += between_duplicates
#             else:
#                 throw_away[genotype] = between_duplicates
#         else: # If it's a genotype not seen in Andersen/NCBI_Virus, don't throw it
#             print("GISAID genotype:", gisaid_genotype["Genotype"].values[0])
#             throw_away[genotype] = []
#     counter += 1

# for key in throw_away:
#     print(key)
#     print(len(throw_away[key]))

# kept_seqs = []
# print(len(segment_fastas))
# for gisaid_df in segment_fastas:
#     # print(len(genotype_group))
#     # for gisaid_df in genotype_group:
#         # print(gisaid_df)
#     if len(gisaid_df["Genotype"].dropna()) > 0: # If there are sequences
#         genotype = gisaid_df["Genotype"].values[0]
#         # print(len(genotype_seq_keep[genotype]))
#         # gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]

#         # Check how much was thrown away
#         if genotype in throw_away:
#             to_throw = throw_away[genotype]
#             print("Original length:", genotype, len(gisaid_df))
#             gisaid_df_new = gisaid_df[~gisaid_df['Identifier'].isin(to_throw)]
#             print("New length:", len(gisaid_df_new))
#             kept_seqs.append(gisaid_df_new)
#         # else:


# # print(len(kept_seqs))

original: 4644
B3.13_NP
new: 4514
original: 4644
B3.13_HA
new: 4514
original: 4644
B3.13_NS
new: 4514
original: 4644
B3.13_PA
new: 4514
original: 4644
B3.13_MP
new: 4514
original: 4644
B3.13_PB2
new: 4514
original: 4644
B3.13_PB1
new: 4514
original: 4644
B3.13_NA
new: 4514


### Create animal reference if needed 

In [28]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print("New animals to add to reference:", different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv", index=False)

['house_sparrow', 'mallard_duck', 'black-crowned_night-heron', 'antofagasta', 'trumpeter_swan', 'larus_belcheri', 'grackle', 'short_billed_gull', 'norway_rat', 'colorado', 'american_goshawk', 'rock_dove', 'black_legged_kittiwake', 'american_green-winged_teal', 'blue-winged_teal', 'sula_variegata', 'common_barn_owl', 'white_winged_scoter', 'chukar', 'cascade_duck', 'buff-necked_ibis', 'striped_skunk', 'wild_mink', 'red-necked_grebe', 'peruvean_booby', 'backyard_chicken', 'silkie_chicken', 'bonapartes_gull', 'american_coot', 'bird', 'gannet', 'double-crested_co', 'thalasseus_maximus', 'amazon_parrot', 'environment', 'long-tailed_duck', 'mountain_lion', 'black_bear', 'roseate_spoonbill', 'avian', 'poultry', 'black-billed_magpie', 'brant', 'black_swan', 'geoffroys_cat', 'quail', 'canvasback', 'tern', 'common_murre', 'american_crow', 'redhead_duck', 'cormorant', 'crested_jay', 'northwestern_crow', 'weasel', 'eagle', 'peregrine_falcon', 'northern_harrier', 'common_loon', 'finch', 'gull_spp.'

In [29]:
# Ensure that user checks if there are any new animals

input("Check animals output. Afterwards, press ESCAPE to continue.")

''

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [30]:
# 16 files needed
# huge_fasta = pd.DataFrame()

# for fastas in gisaid_dict.values():
#     # print(fastas.columns)
#     # print(len(fastas))
#     # break
#     # for f in fastas: # 16 files per batch 
#         # print(f)
#         # break 
#     print(fastas["Genotype"])
#     # Find Clade
#     metadata_fasta = pd.DataFrame()
#     for metadata in all_metadata_files:
#         metadata_fasta = pd.concat([metadata_fasta, metadata])
    
#     metadata_fasta["Identifier"] = metadata_fasta["Isolate_Id"]
#     metadata_fasta_concat = fastas.merge(metadata_fasta, on="Identifier", how="left")

#     huge_fasta = pd.concat([huge_fasta, metadata_fasta_concat])

# print(huge_fasta.columns)



# Now separate huge_fasta into genotypes * segments fastas
big_fastas = []

big_fastas = segment_fastas # If no NCBI Virus/Andersen

# print(huge_fasta)

# print(gisaid_dict)

# genotypes.append("Unassigned")
for gen in genotypes:
    print(gen)
    # for gisaid_fasta in gisaid_dict[gen]:
    
    # print(gen)
    for seg in unique_segments:
        print(seg)
        # gisaid_fasta = gisaid_dict[gen + "_" + seg]
        # seg_specific_fasta = gisaid_fasta[gisaid_fasta["Segment"] == seg]
        print(gisaid_fasta)
        big_fastas.append(gisaid_fasta)

    print(gisaid_fasta[["Genotype"]])



B3.13
NP
                                                  Header         Isolate_Id  \
72     EPI_ISL_19497981|A/California/152/2024|A_/_H5N...                152   
80     EPI_ISL_19497980|A/California/153/2024|A_/_H5N...                153   
82     EPI_ISL_19532150|A/chicken/USA/24-031862-002/2...      24-031862-002   
93     EPI_ISL_19532151|A/dairy_cow/California/031313...         031313-004   
98     EPI_ISL_19532152|A/dairy_cow/California/031313...         031313-003   
...                                                  ...                ...   
38346  EPI_ISL_19167904|A/dairy_cow/Michigan/24_00902...      24_009027-002   
38360  EPI_ISL_19628008|A/California/213/2024|A_/_H5N...                213   
38362  EPI_ISL_19332143|A/american_robin/Iowa/24-0188...      24-018879-003   
38370  EPI_ISL_19167901|A/chicken/Michigan/24-010308-...  24-010308-006-300   
38378  EPI_ISL_19167900|A/chicken/Michigan/24-010308-...  24-010308-001-300   

                                  Isolate_

In [31]:
# print(huge_fasta[huge_fasta["Genotype_x"] == "Not assigned"][["Identifier", "Genotype_x", "Genotype_y","Subtype_x"]]) # "Clade",  "Pathogenicity"]])

# Now that we have x fastas, write the files
for fasta in big_fastas:

    # Fix animals
    fasta = fix_animals(fasta, animals_df)
    # Create a dictionary to create a file
    # fasta_df = fasta[["New_Name", "Sequence", "Clade"]]
    # print(fasta_df[fasta_df["Clade"].isna() == False])
    # break
    # fasta["New_Name_Clade_Path"] = fasta["New_Name"].apply(lambda x: x.split("\n")[0]) # + fasta["Clade"].apply(lambda x: "/" + x if x == x else "") + fasta["Pathogenicity"].apply(lambda x: "/" + x if x == x else "")
    # print(fasta_df["New_Name_Clade"])
    print("Original length:", len(fasta))
    fasta_dedup_ids = fasta.drop_duplicates(subset="Identifier", keep="last")
    # fasta = fasta_dedup_ids.drop_duplicates(subset="Partials", keep="last")
    print("New length:", len(fasta))
    # break 
    # fasta_df = fasta[["New_Name_Clade_Path", "Sequence"]]
    fasta_dict = pd.Series(fasta["Sequence"].values,index=fasta.full_header).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = gisaid_files + fasta["Genotype"].values[0].split(" ")[0] + "_" + fasta["Segment"].values[0] + "_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            # print(item)
            # elif "EPI_ISL_20151596" in item:
            #     item = "EPI_ISL_20151596|A/Brown_Skua/Gough_Island/047354/2024|H5N1|Antarctica|2024-09-20|wild_avian|B3.2"
            output_file.write(item + "\n")
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

# print(len(huge_fasta))
print(len(big_fastas[0]))

KeyError: 'Isolate_Name'

## Concatenate to Andersen_NCBI files and save

In [ ]:

# Concat
os.chdir(complete_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

for gisaid_fasta_key in gisaid_dict:
    print(gisaid_fasta_key)
    for andersen_ncbi_fasta_key in andersen_ncbi_genotypes:
        # print(andersen_ncbi_fasta_key)
        if gisaid_fasta_key == andersen_ncbi_fasta_key: # If the genotypes/segments are the same
            print(andersen_ncbi_fasta_key)
            gisaid_fasta = gisaid_dict[gisaid_fasta_key]
            andersen_ncbi_fasta = andersen_ncbi_genotypes[andersen_ncbi_fasta_key]
            # print(gisaid_fasta)
            print(andersen_ncbi_fasta.columns)
            # andersen_ncbi_fasta["full_header"] = andersen_ncbi_fasta["Header"].apply(lambda x: ">" + x if ">" not in x else x)
            combined_fasta = pd.concat([gisaid_fasta, andersen_ncbi_fasta], ignore_index=True).drop_duplicates(subset=["Partials", "Year"], keep="last")
            combined_fasta = combined_fasta.rename(columns={"Sequence":"sequence"})
            # combined_fasta["Header"] = combined_fasta["Header"].apply(lambda x: ">" + x if ">" not in x else x)
            # combined_fasta["full_header"] = combined_fasta["full_header"].fillna(combined_fasta["Header"])
            combined_fasta["full_header"] = combined_fasta["full_header"].apply(lambda x: ">" + x if ">" not in x else x)
            # print(combined_fasta)
            # print(combined_fasta.columns)
            print(combined_fasta) #[~combined_fasta["full_header"].str.contains(">")])
            # try:
            df_to_fasta(combined_fasta, gisaid_fasta_key + "_" + date_range + ".fasta", Path(complete_files))
            # except:
            #     print("Andersen key:", andersen_ncbi_fasta_key)


B3.13_NP
B3.13_NP
Index(['Header', 'Isolate_Id', 'Isolate_Name', 'Subtype', 'Partials',
       'Location', 'Geo_Location', 'Date Collected', 'Species', 'Host_Type',
       'Genotype', 'Sequence', 'Identifier', 'full_header', 'Year', 'Segment'],
      dtype='object')
                                                 Header     Isolate_Id  \
0     EPI_ISL_19497981|A/California/152/2024|A_/_H5N...            152   
1     EPI_ISL_19497980|A/California/153/2024|A_/_H5N...            153   
79    EPI_ISL_19531293|A/California/151/2024|A_/_H5N...            151   
80    EPI_ISL_19628600|A/macaque/Montana/60_LLL/2024...         60_LLL   
81    EPI_ISL_19628594|A/macaque/Montana/51_LML/2024...         51_LML   
...                                                 ...            ...   
9161  SRR28752446|A/blackbird/Texas/24-008354-001/20...  24-008354-001   
9162  GCA_039158345.1|A/Bovine/texas/24-029328-01/20...   24-029328-01   
9163  GCA_039158355.1|A/bovine/texas/24-029328-02/20...   24-029328